# Fabric Defect Dataset — Inspection / Audit

Standalone dataset audit for `fabric_defect_detection_MERGED_CLEAN_4CLASS.zip`.

- No GPU required.
- Does **not** modify the original dataset — read-only.
- Auto-detects the zip, train/valid/test folders, and COCO JSON files —
  nothing about the layout is assumed.

Checks performed: train/valid/test + JSON detection, image/annotation counts,
category IDs and names, image dimensions, missing images, missing
annotations, invalid bounding boxes, duplicate images/annotations, empty
annotations, class distribution, objects per image, and cross-split leakage
(identical images appearing in more than one split).

**Usage on Kaggle:** attach your dataset as a data source, then run all cells.


In [ ]:
%%writefile 01_inspect_dataset.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
01_inspect_dataset.py
=====================
STANDALONE dataset inspection / audit for the fabric defect detection dataset.

Run this FIRST, before training. It needs no GPU.

It does NOT modify the original dataset. It only reads, and writes a report to
/kaggle/working/faster_rcnn_results/dataset_audit.txt (+ .json)

Usage on Kaggle:
    !python 01_inspect_dataset.py
or paste into a notebook cell and call main().
"""

import os
import io
import sys
import json
import time
import zipfile
import hashlib
import platform
from collections import Counter, defaultdict

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", ".", "/content", "/mnt/data"]
EXTRACT_DIR = "/kaggle/working/dataset_extracted"
OUT_DIR = "/kaggle/working/faster_rcnn_results"
EXPECTED_CLASS_KEYWORDS = ["hole", "knot", "stain"]
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

# hash-based duplicate detection can be slow on huge sets; cap it
DUPLICATE_CHECK_MAX_IMAGES = 20000


# ----------------------------------------------------------------------------
# small utils
# ----------------------------------------------------------------------------
def log(msg=""):
    print(msg, flush=True)


def hr(title=None):
    log("=" * 78)
    if title:
        log(title)
        log("=" * 78)


def human(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024.0:
            return f"{n:3.1f}{unit}"
        n /= 1024.0
    return f"{n:.1f}PB"


# ----------------------------------------------------------------------------
# 1. locate + extract the dataset
# ----------------------------------------------------------------------------
def find_zip_candidates(roots=SEARCH_ROOTS):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            # do not descend into our own extraction dir
            if os.path.abspath(dirpath).startswith(os.path.abspath(EXTRACT_DIR)):
                dirnames[:] = []
                continue
            for fn in filenames:
                if fn.lower().endswith(".zip"):
                    found.append(os.path.join(dirpath, fn))
    return sorted(set(found))


def find_coco_jsons(root):
    """Walk a directory and return every JSON that looks like COCO detection format."""
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        for fn in filenames:
            if not fn.lower().endswith(".json"):
                continue
            path = os.path.join(dirpath, fn)
            try:
                if os.path.getsize(path) > 500 * 1024 * 1024:
                    continue
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    data = json.load(f)
            except Exception:
                continue
            if isinstance(data, dict) and "images" in data and "annotations" in data \
                    and "categories" in data:
                hits.append(path)
    return sorted(hits)


def guess_split(path):
    """Infer train/valid/test from the path string."""
    p = path.replace("\\", "/").lower()
    parts = p.split("/")
    # look at the deepest components first
    for seg in reversed(parts):
        if "train" in seg:
            return "train"
        if "valid" in seg or seg == "val" or "validation" in seg or seg.startswith("val_"):
            return "valid"
        if "test" in seg:
            return "test"
    return None


def locate_images_dir(json_path, coco):
    """
    Find the directory that actually contains the images referenced by this
    COCO json. Tries: json's own dir, ./images, parent, parent/images.
    """
    names = [im.get("file_name", "") for im in coco.get("images", [])][:50]
    names = [os.path.basename(n) for n in names if n]
    if not names:
        return os.path.dirname(json_path)

    jdir = os.path.dirname(json_path)
    candidates = [
        jdir,
        os.path.join(jdir, "images"),
        os.path.dirname(jdir),
        os.path.join(os.path.dirname(jdir), "images"),
    ]
    best, best_hits = jdir, -1
    for c in candidates:
        if not os.path.isdir(c):
            continue
        hits = sum(1 for n in names if os.path.isfile(os.path.join(c, n)))
        if hits > best_hits:
            best, best_hits = c, hits
    if best_hits > 0:
        return best

    # fall back: full recursive search for the first filename
    search_root = os.path.dirname(jdir) or jdir
    target = names[0]
    for dirpath, _, filenames in os.walk(search_root):
        if target in filenames:
            return dirpath
    return jdir


def discover_dataset(zip_path=None, verbose=True):
    """
    Returns dict:
      { 'root': str,
        'splits': { 'train': {'json': path, 'images_dir': path}, ... } }
    """
    root = None

    # (a) explicit / auto zip
    zips = [zip_path] if zip_path else find_zip_candidates()
    if verbose and zips:
        log("ZIP candidates found:")
        for z in zips:
            log(f"   {z}  ({human(os.path.getsize(z))})")

    # prefer a zip whose name mentions fabric/defect
    chosen_zip = None
    for z in zips:
        if z and os.path.isfile(z):
            base = os.path.basename(z).lower()
            if "fabric" in base or "defect" in base:
                chosen_zip = z
                break
    if chosen_zip is None and zips and zips[0] and os.path.isfile(zips[0]):
        chosen_zip = zips[0]

    if chosen_zip:
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        marker = os.path.join(EXTRACT_DIR, "._extracted_" +
                              hashlib.md5(chosen_zip.encode()).hexdigest()[:10])
        if os.path.exists(marker):
            if verbose:
                log(f"\nAlready extracted -> {EXTRACT_DIR}")
        else:
            if verbose:
                log(f"\nExtracting {chosen_zip} -> {EXTRACT_DIR} ...")
            t0 = time.time()
            with zipfile.ZipFile(chosen_zip) as zf:
                zf.extractall(EXTRACT_DIR)
            open(marker, "w").write("ok")
            if verbose:
                log(f"Extracted in {time.time()-t0:.1f}s")
        root = EXTRACT_DIR

    # (b) no zip -> look for an already-extracted dataset dir
    if root is None:
        for r in SEARCH_ROOTS:
            if os.path.isdir(r) and find_coco_jsons(r):
                root = r
                break

    if root is None:
        raise FileNotFoundError(
            "Could not locate the dataset. Put the ZIP (or the extracted folder) "
            "somewhere under /kaggle/input or /kaggle/working, or pass zip_path=..."
        )

    jsons = find_coco_jsons(root)
    if not jsons:
        raise FileNotFoundError(f"No COCO-format JSON found under {root}")

    if verbose:
        log(f"\nCOCO JSON files detected under {root}:")
        for j in jsons:
            log(f"   {j}")

    splits = {}
    unknown = []
    for j in jsons:
        s = guess_split(j)
        if s is None:
            unknown.append(j)
            continue
        # if two jsons map to the same split, keep the one with more images
        with open(j, "r", encoding="utf-8", errors="ignore") as f:
            data = json.load(f)
        n_img = len(data.get("images", []))
        if s in splits and splits[s]["n_images"] >= n_img:
            continue
        splits[s] = {
            "json": j,
            "images_dir": locate_images_dir(j, data),
            "n_images": n_img,
        }

    if unknown and verbose:
        log("\n[warn] JSON files whose split could not be inferred from the path:")
        for u in unknown:
            log(f"   {u}")

    missing = [s for s in ("train", "valid", "test") if s not in splits]
    if "valid" in missing and "val" in splits:
        splits["valid"] = splits.pop("val")
        missing.remove("valid")
    if missing and verbose:
        log(f"\n[warn] splits not found: {missing}")

    return {"root": root, "splits": splits}


# ----------------------------------------------------------------------------
# 2. category mapping
# ----------------------------------------------------------------------------
def build_category_mapping(categories, verbose=True):
    """
    Map COCO category ids -> contiguous model labels:
        0 background, 1 hole, 2 knot, 3 stain
    Matching is done on normalised names; falls back to sorted-id order with a
    loud warning if the names are unrecognisable.
    """
    norm = []
    for c in categories:
        name = str(c.get("name", "")).strip().lower().replace("-", "_").replace(" ", "_")
        norm.append((c["id"], name, c.get("name")))

    coco_id_to_contig = {}
    contig_to_name = {}
    used = set()
    unmatched = []

    for cid, name, raw in norm:
        hit = None
        for i, kw in enumerate(EXPECTED_CLASS_KEYWORDS):
            if kw in name:
                hit = i + 1
                break
        if hit is not None and hit not in used:
            coco_id_to_contig[cid] = hit
            contig_to_name[hit] = EXPECTED_CLASS_KEYWORDS[hit - 1]
            used.add(hit)
        else:
            unmatched.append((cid, raw))

    if unmatched:
        if verbose:
            log("\n[warn] categories not matched by name: " + str(unmatched))
        free = [i for i in (1, 2, 3) if i not in used]
        for (cid, raw), slot in zip(sorted(unmatched), free):
            coco_id_to_contig[cid] = slot
            contig_to_name[slot] = str(raw)
            used.add(slot)

    # COCO supercategory rows sometimes carry id 0 with no real annotations;
    # anything still unmapped is recorded so the audit can flag it.
    contig_to_coco_id = {v: k for k, v in coco_id_to_contig.items()}
    return coco_id_to_contig, contig_to_coco_id, contig_to_name


# ----------------------------------------------------------------------------
# 3. audit one split
# ----------------------------------------------------------------------------
def audit_split(split_name, json_path, images_dir, do_hash=True, verbose=True):
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None

    with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
        coco = json.load(f)

    images = coco.get("images", [])
    anns = coco.get("annotations", [])
    cats = coco.get("categories", [])

    coco_id_to_contig, contig_to_coco_id, contig_to_name = build_category_mapping(cats, verbose=False)

    rep = {
        "split": split_name,
        "json": json_path,
        "images_dir": images_dir,
        "n_images_declared": len(images),
        "n_annotations_declared": len(anns),
        "categories": [{"id": c["id"], "name": c.get("name"),
                        "contiguous_label": coco_id_to_contig.get(c["id"])} for c in cats],
    }

    img_by_id = {}
    duplicate_image_ids = []
    for im in images:
        if im["id"] in img_by_id:
            duplicate_image_ids.append(im["id"])
        img_by_id[im["id"]] = im

    # --- files on disk ---
    missing_files = []
    corrupted = []
    dim_mismatch = []
    sizes = Counter()
    widths, heights = [], []
    hashes = defaultdict(list)

    for im in images:
        fn = os.path.basename(im.get("file_name", ""))
        path = os.path.join(images_dir, fn)
        if not os.path.isfile(path):
            missing_files.append(fn)
            continue
        try:
            with Image.open(path) as pil:
                pil.verify()
            with Image.open(path) as pil:
                w, h = pil.size
        except Exception as e:
            corrupted.append((fn, repr(e)[:120]))
            continue
        widths.append(w); heights.append(h)
        sizes[(w, h)] += 1
        jw, jh = im.get("width"), im.get("height")
        if jw and jh and (jw != w or jh != h):
            dim_mismatch.append({"file": fn, "json": [jw, jh], "actual": [w, h]})
        if do_hash and len(images) <= DUPLICATE_CHECK_MAX_IMAGES:
            try:
                with open(path, "rb") as fh:
                    hashes[hashlib.md5(fh.read()).hexdigest()].append(fn)
            except Exception:
                pass

    duplicate_files = {h: fs for h, fs in hashes.items() if len(fs) > 1}

    # --- annotations ---
    invalid_boxes = []
    clipped_boxes = []
    bad_category = []
    crowd = 0
    per_class = Counter()
    per_image_count = Counter()
    areas = []
    aspect = []
    seen_ann = set()
    duplicate_anns = []
    valid_anns = 0

    for a in anns:
        aid = a.get("id")
        img_id = a.get("image_id")
        cid = a.get("category_id")
        bbox = a.get("bbox")

        key = (img_id, cid, tuple(round(float(v), 2) for v in bbox) if bbox else None)
        if key in seen_ann:
            duplicate_anns.append(aid)
        seen_ann.add(key)

        if cid not in coco_id_to_contig:
            bad_category.append({"ann_id": aid, "category_id": cid})
            continue
        if a.get("iscrowd", 0):
            crowd += 1
        if img_id not in img_by_id:
            invalid_boxes.append({"ann_id": aid, "reason": "image_id not in images"})
            continue
        if not bbox or len(bbox) != 4:
            invalid_boxes.append({"ann_id": aid, "reason": "bbox malformed"})
            continue

        x, y, w, h = [float(v) for v in bbox]
        im = img_by_id[img_id]
        W = float(im.get("width") or 0) or 1e9
        H = float(im.get("height") or 0) or 1e9

        if w <= 0 or h <= 0:
            invalid_boxes.append({"ann_id": aid, "reason": f"non-positive w/h ({w:.2f},{h:.2f})"})
            continue
        x1, y1, x2, y2 = x, y, x + w, y + h
        if x1 < -1 or y1 < -1 or x2 > W + 1 or y2 > H + 1:
            clipped_boxes.append({"ann_id": aid, "box": [x1, y1, x2, y2], "img": [W, H]})
        cx1, cy1 = max(0.0, x1), max(0.0, y1)
        cx2, cy2 = min(W, x2), min(H, y2)
        if cx2 - cx1 <= 1e-3 or cy2 - cy1 <= 1e-3:
            invalid_boxes.append({"ann_id": aid, "reason": "zero area after clipping"})
            continue

        valid_anns += 1
        lab = coco_id_to_contig[cid]
        per_class[contig_to_name.get(lab, str(lab))] += 1
        per_image_count[img_id] += 1
        bw, bh = cx2 - cx1, cy2 - cy1
        areas.append(bw * bh)
        aspect.append(bw / max(bh, 1e-6))

    empty_images = [im["id"] for im in images if per_image_count.get(im["id"], 0) == 0]

    def pct(vals, q):
        if not vals:
            return 0.0
        s = sorted(vals)
        i = min(len(s) - 1, max(0, int(round(q * (len(s) - 1)))))
        return float(s[i])

    rep.update({
        "n_images_on_disk": len(images) - len(missing_files) - len(corrupted),
        "missing_image_files": missing_files,
        "corrupted_images": corrupted,
        "duplicate_image_ids": duplicate_image_ids,
        "duplicate_image_files": duplicate_files,
        "json_vs_actual_dimension_mismatch": dim_mismatch[:50],
        "n_dimension_mismatch": len(dim_mismatch),
        "unique_resolutions": sorted(
            [{"wh": list(k), "count": v} for k, v in sizes.items()],
            key=lambda d: -d["count"])[:20],
        "width_min_med_max": [min(widths) if widths else 0, pct(widths, 0.5),
                              max(widths) if widths else 0],
        "height_min_med_max": [min(heights) if heights else 0, pct(heights, 0.5),
                               max(heights) if heights else 0],
        "n_valid_annotations": valid_anns,
        "n_invalid_boxes": len(invalid_boxes),
        "invalid_boxes_sample": invalid_boxes[:25],
        "n_boxes_needing_clip": len(clipped_boxes),
        "clipped_boxes_sample": clipped_boxes[:25],
        "n_bad_category": len(bad_category),
        "bad_category_sample": bad_category[:25],
        "n_iscrowd": crowd,
        "n_duplicate_annotations": len(duplicate_anns),
        "class_distribution": dict(per_class),
        "n_images_without_annotations": len(empty_images),
        "objects_per_image": {
            "mean": round(sum(per_image_count.values()) / max(1, len(images)), 3),
            "max": max(per_image_count.values()) if per_image_count else 0,
            "p50": pct(list(per_image_count.values()) + [0] * len(empty_images), 0.50),
            "p95": pct(list(per_image_count.values()) + [0] * len(empty_images), 0.95),
        },
        "box_area_px": {
            "p05": round(pct(areas, 0.05), 1), "p50": round(pct(areas, 0.50), 1),
            "p95": round(pct(areas, 0.95), 1),
            "min": round(min(areas), 1) if areas else 0,
            "max": round(max(areas), 1) if areas else 0,
        },
        "box_aspect_ratio_w_over_h": {
            "p05": round(pct(aspect, 0.05), 3), "p50": round(pct(aspect, 0.50), 3),
            "p95": round(pct(aspect, 0.95), 3),
        },
        "small_objects_lt_32x32": int(sum(1 for a in areas if a < 32 * 32)),
        "tiny_objects_lt_16x16": int(sum(1 for a in areas if a < 16 * 16)),
    })
    return rep


def print_report(rep):
    hr(f"SPLIT: {rep['split'].upper()}")
    log(f"  json            : {rep['json']}")
    log(f"  images_dir      : {rep['images_dir']}")
    log(f"  images (json)   : {rep['n_images_declared']}")
    log(f"  images on disk  : {rep['n_images_on_disk']}")
    log(f"  annotations     : {rep['n_annotations_declared']} declared / "
        f"{rep['n_valid_annotations']} valid")
    log("")
    log("  categories:")
    for c in rep["categories"]:
        log(f"     coco_id={c['id']:<4} name={str(c['name']):<12} -> label {c['contiguous_label']}")
    log("")
    log("  class distribution (valid annotations):")
    total = max(1, sum(rep["class_distribution"].values()))
    for k, v in sorted(rep["class_distribution"].items(), key=lambda x: -x[1]):
        log(f"     {k:<10} {v:>7}   ({100.0*v/total:5.2f}%)")
    log("")
    log(f"  images without annotations : {rep['n_images_without_annotations']}")
    log(f"  objects/image              : mean={rep['objects_per_image']['mean']} "
        f"p50={rep['objects_per_image']['p50']} p95={rep['objects_per_image']['p95']} "
        f"max={rep['objects_per_image']['max']}")
    log(f"  box area px (p05/p50/p95)  : {rep['box_area_px']['p05']} / "
        f"{rep['box_area_px']['p50']} / {rep['box_area_px']['p95']}")
    log(f"  boxes < 32x32 px           : {rep['small_objects_lt_32x32']}")
    log(f"  boxes < 16x16 px           : {rep['tiny_objects_lt_16x16']}")
    log(f"  aspect ratio w/h p05/50/95 : {rep['box_aspect_ratio_w_over_h']['p05']} / "
        f"{rep['box_aspect_ratio_w_over_h']['p50']} / {rep['box_aspect_ratio_w_over_h']['p95']}")
    log(f"  width  min/med/max         : {rep['width_min_med_max']}")
    log(f"  height min/med/max         : {rep['height_min_med_max']}")
    log("")
    log("  ISSUES")
    log(f"     missing image files     : {len(rep['missing_image_files'])}")
    log(f"     corrupted images        : {len(rep['corrupted_images'])}")
    log(f"     duplicate image ids     : {len(rep['duplicate_image_ids'])}")
    log(f"     duplicate image files   : {len(rep['duplicate_image_files'])}")
    log(f"     dimension mismatches    : {rep['n_dimension_mismatch']}")
    log(f"     invalid boxes (dropped) : {rep['n_invalid_boxes']}")
    log(f"     boxes needing clipping  : {rep['n_boxes_needing_clip']}")
    log(f"     bad category ids        : {rep['n_bad_category']}")
    log(f"     iscrowd annotations     : {rep['n_iscrowd']}")
    log(f"     duplicate annotations   : {rep['n_duplicate_annotations']}")
    if rep["missing_image_files"][:5]:
        log(f"     e.g. missing: {rep['missing_image_files'][:5]}")
    if rep["invalid_boxes_sample"][:3]:
        log(f"     e.g. invalid: {rep['invalid_boxes_sample'][:3]}")
    log("")


def check_cross_split_leakage(reports):
    """Same image file appearing in more than one split is a leak."""
    hr("CROSS-SPLIT LEAKAGE CHECK (identical file content across splits)")
    by_split = {}
    for r in reports:
        h = {}
        for hh, files in r.get("duplicate_image_files", {}).items():
            h[hh] = files
        by_split[r["split"]] = r
    # we need full hash sets -> recompute cheaply from disk
    import hashlib as _h
    split_hashes = {}
    for r in reports:
        d = r["images_dir"]
        hs = {}
        if os.path.isdir(d):
            for fn in os.listdir(d):
                if fn.lower().endswith(IMG_EXTS):
                    p = os.path.join(d, fn)
                    try:
                        with open(p, "rb") as f:
                            hs.setdefault(_h.md5(f.read()).hexdigest(), []).append(fn)
                    except Exception:
                        pass
        split_hashes[r["split"]] = hs

    names = list(split_hashes.keys())
    leaks = {}
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            common = set(split_hashes[a]) & set(split_hashes[b])
            if common:
                leaks[f"{a}|{b}"] = [
                    (split_hashes[a][c][0], split_hashes[b][c][0]) for c in list(common)[:20]
                ]
                log(f"  [LEAK] {len(common)} identical images shared between {a} and {b}")
    if not leaks:
        log("  OK - no identical images shared across splits.")
    log("")
    return leaks


def main(zip_path=None):
    t0 = time.time()
    os.makedirs(OUT_DIR, exist_ok=True)

    hr("ENVIRONMENT")
    log(f"  python   : {platform.python_version()}")
    log(f"  platform : {platform.platform()}")
    try:
        from PIL import Image as _I
        log(f"  Pillow   : {_I.__version__}")
    except Exception:
        log("  Pillow   : NOT AVAILABLE (pip install pillow)")
    log("")

    hr("DATASET DISCOVERY")
    info = discover_dataset(zip_path=zip_path)
    log(f"\nResolved root: {info['root']}")
    for s, d in info["splits"].items():
        log(f"  {s:<6} json={d['json']}")
        log(f"  {'':<6} imgs={d['images_dir']}")
    log("")

    reports = []
    for split in ("train", "valid", "test"):
        if split not in info["splits"]:
            log(f"[warn] split '{split}' missing - skipped")
            continue
        d = info["splits"][split]
        rep = audit_split(split, d["json"], d["images_dir"])
        print_report(rep)
        reports.append(rep)

    leaks = check_cross_split_leakage(reports)

    hr("SUMMARY")
    log(f"{'split':<8}{'images':>9}{'anns':>9}{'hole':>8}{'knot':>8}{'stain':>8}{'empty':>8}")
    for r in reports:
        cd = r["class_distribution"]
        log(f"{r['split']:<8}{r['n_images_declared']:>9}{r['n_valid_annotations']:>9}"
            f"{cd.get('hole',0):>8}{cd.get('knot',0):>8}"
            f"{cd.get('stain',0):>8}{r['n_images_without_annotations']:>8}")
    log("")

    out_json = os.path.join(OUT_DIR, "dataset_audit.json")
    with open(out_json, "w") as f:
        json.dump({"root": info["root"],
                   "splits": {k: {kk: vv for kk, vv in v.items()} for k, v in info["splits"].items()},
                   "reports": reports,
                   "cross_split_leaks": leaks}, f, indent=2, default=str)
    log(f"Audit JSON written to {out_json}")
    log(f"Done in {time.time()-t0:.1f}s")
    return info, reports


if __name__ == "__main__":
    zp = sys.argv[1] if len(sys.argv) > 1 else None
    main(zp)


## Run the audit

Searches `/kaggle/input`, `/kaggle/working`, and the current directory for the
zip (or an already-extracted dataset), extracts it if needed, finds every
COCO JSON, infers which split it belongs to from its path, and locates the
matching images folder by checking that the referenced filenames actually
exist on disk.

Writes a full report to stdout and to
`/kaggle/working/faster_rcnn_results/dataset_audit.json`.

In [ ]:
!python 01_inspect_dataset.py


## (Optional) Inspect the audit programmatically

Load the JSON report back in and pull out whatever you need — e.g. to decide
on `min_size`/`max_size` or anchor aspect ratios before training.

In [ ]:
import json, os

path = "/kaggle/working/faster_rcnn_results/dataset_audit.json"
if os.path.exists(path):
    with open(path) as f:
        audit = json.load(f)
    for rep in audit["reports"]:
        print(f"--- {rep['split']} ---")
        print("  images:", rep["n_images_declared"], "| valid annotations:", rep["n_valid_annotations"])
        print("  class distribution:", rep["class_distribution"])
        print("  images without annotations:", rep["n_images_without_annotations"])
        print("  box area p05/p50/p95:", rep["box_area_px"])
        print("  aspect ratio p05/p50/p95:", rep["box_aspect_ratio_w_over_h"])
        print()
else:
    print("Run the audit cell above first.")
